# Preprocessing Melhorado - MIQR-CC Dataset

Este notebook estende o pipeline de preprocessing da baseline original com:
- **CLAHE** (Contrast Limited Adaptive Histogram Equalization) para realçar estruturas biliares
- **Padding com aspect ratio preservado** antes do resize
- **Normalização** de intensidade
- **Data Augmentation** para treino (rotações, flips, jitter de contraste)
- **Pipeline unificado** pronto a integrar com PyTorch DataLoader

A lógica de segmentação por linhas verticais (Canny + Hough) é mantida da baseline:
> Martins et al., *MIQR-CC Dataset*, https://github.com/monicaccmartins/MIQR-CC-Dataset

In [2]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

---
## 1. Segmentação por linhas verticais (baseline original)

Mantida sem alterações da baseline. Deteta linhas verticais com Canny + Hough Transform
e corta a imagem em segmentos, separando painéis laterais de fluoroscopia.

In [3]:
def detect_vertical_lines(image, canny1=30, canny2=150, hough_threshold=120,
                           min_line_length=70, max_line_gap=4, angle_threshold_deg=10):
    """Deteta linhas verticais via Canny + HoughLinesP. (baseline original)"""
    edges = cv2.Canny(image, canny1, canny2)
    lines = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi / 180,
        threshold=hough_threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )
    vertical_lines = []
    if lines is not None:
        for l in lines:
            x1, y1, x2, y2 = l[0]
            angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
            if abs(angle - 90) <= angle_threshold_deg:
                vertical_lines.append(l)
    return edges, vertical_lines


def segment_image_by_vertical_lines(image, lines, min_width=30):
    """Corta a imagem nos x das linhas verticais detetadas. (baseline original)"""
    if not lines:
        return [image]
    lines = sorted(lines, key=lambda x: x[0][0])
    x_positions = [0] + [l[0][0] for l in lines] + [image.shape[1]]
    segments = []
    for i in range(len(x_positions) - 1):
        x_start, x_end = x_positions[i], x_positions[i + 1]
        if x_end - x_start >= min_width:
            segments.append(image[:, x_start:x_end])
    return segments if segments else [image]


def is_image_valid(image, min_width=212, black_percentile=90, contrast_threshold=10):
    """Valida segmento por largura, escuridão e contraste. (baseline original)"""
    h, w = image.shape
    if w < min_width:
        return False
    if np.percentile(image, black_percentile) < 1:
        return False
    if image.max() - image.min() < contrast_threshold:
        return False
    return True

---
## 2. Melhorias: CLAHE + Padding + Normalização

Estas funções são aplicadas **após** a segmentação da baseline.

In [4]:
def apply_clahe(image, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Aplica CLAHE (Contrast Limited Adaptive Histogram Equalization).
    Melhora o contraste local - especialmente útil para imagens de fluoroscopia
    onde cálculos e estenoses têm contraste subtil.
    
    clip_limit: limita amplificação de ruído (2.0 é um bom balanço)
    tile_grid:  tamanho das tiles locais (8x8 é o padrão)
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(image)


def pad_to_square(image):
    """
    Adiciona padding preto para tornar a imagem quadrada,
    preservando o aspect ratio original.
    Evita distorção dos ductos biliares no resize final.
    """
    h, w = image.shape[:2]
    size = max(h, w)
    top    = (size - h) // 2
    bottom = size - h - top
    left   = (size - w) // 2
    right  = size - w - left
    return cv2.copyMakeBorder(image, top, bottom, left, right,
                               borderType=cv2.BORDER_CONSTANT, value=0)


def resize_image(image, target_size=300):
    """
    Redimensiona para target_size x target_size.
    EfficientNet-B3 espera 300x300; ResNet-50 espera 224x224.
    """
    return cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)


def normalize_image(image):
    """
    Normaliza para [0, 1] e depois aplica média/std típicos de imagens médicas grayscale.
    Retorna array float32.
    """
    img = image.astype(np.float32) / 255.0
    mean = 0.485
    std  = 0.229
    img = (img - mean) / std
    return img

---
## 3. Pipeline unificado

Junta a segmentação da baseline com as melhorias.
Retorna o melhor segmento válido já pronto para o modelo.

In [5]:
def preprocess_image(image_path, target_size=300, apply_norm=True):
    """
    Pipeline completo:
      1. Lê imagem em grayscale
      2. Deteta linhas verticais e segmenta (baseline)
      3. Seleciona o segmento válido mais largo
      4. Aplica CLAHE
      5. Padding para quadrado + resize
      6. Normalização (opcional, desativar para visualização)
    
    Retorna: np.ndarray float32 shape (target_size, target_size)
             ou None se não encontrar segmento válido
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"❌ Erro ao abrir: {image_path}")
        return None

    # --- Baseline: segmentação ---
    edges, lines = detect_vertical_lines(img)
    segments     = segment_image_by_vertical_lines(img, lines)
    valid_segs   = [s for s in segments if is_image_valid(s)]

    # Seleciona o segmento válido mais largo
    if valid_segs:
        seg = max(valid_segs, key=lambda s: s.shape[1])
    else:
        seg = img  # fallback: usa imagem inteira

    # --- Melhorias ---
    seg = apply_clahe(seg)
    seg = pad_to_square(seg)
    seg = resize_image(seg, target_size)

    if apply_norm:
        seg = normalize_image(seg)

    return seg

---
## 4. Data Augmentation (apenas para treino)

Transforms do torchvision aplicados durante o treino para aumentar a diversidade
do dataset e combater o desequilíbrio de classes.

In [6]:
# Tamanho de input para EfficientNet-B3
# Muda para 224 se usares ResNet-50
INPUT_SIZE = 300

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # modelos pré-treinados esperam 3 canais
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.4,   # contraste é o mais relevante em fluoroscopia
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean (para modelos pré-treinados)
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

---
## 5. Dataset PyTorch

Classe `CPREDataset` pronta a usar com DataLoader.
Aplica o pipeline de preprocessing e os transforms de augmentation.

In [7]:
CLASS_NAMES = ['Biliary_Leaks', 'Lithiasis', 'Normal', 'Stricture']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}


class CPREDataset(Dataset):
    """
    Dataset PyTorch para o MIQR-CC.
    
    Parâmetros:
        df          : DataFrame com colunas 'image_path' e 'label'
        transform   : transforms a aplicar (train ou val/test)
        target_size : tamanho do input do modelo
    """

    def __init__(self, df, transform=None, target_size=300):
        self.df          = df.reset_index(drop=True)
        self.transform   = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        image_path = row['image_path']
        label      = CLASS_TO_IDX[row['label']]

        # Pipeline de preprocessing (sem normalização final — feita pelo transform)
        img = preprocess_image(image_path, target_size=self.target_size, apply_norm=False)

        if img is None:
            # Fallback: imagem preta se o preprocessing falhar
            img = np.zeros((self.target_size, self.target_size), dtype=np.uint8)

        # Converte para PIL para usar com torchvision transforms
        img_pil = Image.fromarray(img.astype(np.uint8))

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        return img_tensor, label

---
## 6. Class Weights para lidar com desequilíbrio

Datasets médicos são tipicamente desbalanceados.
Calculamos pesos inversamente proporcionais à frequência de cada classe.

In [8]:
def compute_class_weights(df, class_col='label'):
    """
    Calcula pesos de classe para CrossEntropyLoss ponderada.
    Retorna tensor com pesos na ordem de CLASS_NAMES.
    """
    counts = df[class_col].value_counts()
    total  = len(df)
    weights = []
    for cls in CLASS_NAMES:
        n = counts.get(cls, 1)
        weights.append(total / (len(CLASS_NAMES) * n))
    
    weight_tensor = torch.tensor(weights, dtype=torch.float32)
    print("Class weights:")
    for cls, w in zip(CLASS_NAMES, weights):
        print(f"  {cls:20s}: {w:.4f}")
    return weight_tensor

---
## 7. Visualização comparativa

Compara o resultado do preprocessing da baseline com o pipeline melhorado.

In [9]:
def visualize_preprocessing(image_path, target_size=300):
    """
    Mostra lado a lado: original | baseline (crop) | + CLAHE | + padding/resize
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"❌ Erro ao abrir: {image_path}")
        return

    # Baseline: segmentação
    edges, lines = detect_vertical_lines(img)
    segments     = segment_image_by_vertical_lines(img, lines)
    valid_segs   = [s for s in segments if is_image_valid(s)]
    seg_baseline = max(valid_segs, key=lambda s: s.shape[1]) if valid_segs else img

    # Melhorias progressivas
    seg_clahe    = apply_clahe(seg_baseline)
    seg_padded   = pad_to_square(seg_clahe)
    seg_resized  = resize_image(seg_padded, target_size)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    titles = ['Original', 'Baseline (crop)', '+ CLAHE', '+ Padding + Resize']
    images = [img, seg_baseline, seg_clahe, seg_resized]

    for ax, title, image in zip(axes, titles, images):
        ax.imshow(image, cmap='gray')
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.suptitle(os.path.basename(image_path), fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 8. Exemplo de uso

Adapta os paths ao teu ambiente local.

In [14]:
METADATA_PATH = os.environ.get("METADATA_PATH", "./data/metadata.csv")
TARGET_SIZE   = 300
DATA_DIR      = "./data"

df = pd.read_csv(METADATA_PATH)
df = df[df['Keep'] == 'Keep']
df = df[df['Label'] != 'Unlabelled']
df = df.rename(columns={'Label': 'label', 'processed_image_path': 'image_path'})
df['label'] = df['label'].replace({
    'Benign Stricture':    'Stricture',
    'Malignant Stricture': 'Stricture',
    'Biliary Leaks':       'Biliary_Leaks'
})
df['image_path'] = df['image_path'].apply(lambda p: os.path.join(DATA_DIR, p))
df = df.reset_index(drop=True)

print(f"Total: {len(df)}")
print(df['label'].value_counts())

Total: 1568
label
Lithiasis        726
Stricture        392
Normal           299
Biliary_Leaks    151
Name: count, dtype: int64


In [15]:
# --- Class weights ---
class_weights = compute_class_weights(df)

# --- Split treino / validação / teste ---
# Adapta se o dataset já vier com split definido
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
df_val,   df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp['label'], random_state=42)

print(f"Treino: {len(df_train)} | Validação: {len(df_val)} | Teste: {len(df_test)}")

# --- Datasets e DataLoaders ---
train_dataset = CPREDataset(df_train, transform=train_transforms, target_size=TARGET_SIZE)
val_dataset   = CPREDataset(df_val,   transform=val_test_transforms, target_size=TARGET_SIZE)
test_dataset  = CPREDataset(df_test,  transform=val_test_transforms, target_size=TARGET_SIZE)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print("\nDataLoaders prontos!")
print(f"Batches treino: {len(train_loader)} | Validação: {len(val_loader)} | Teste: {len(test_loader)}")

Class weights:
  Biliary_Leaks       : 2.5960
  Lithiasis           : 0.5399
  Normal              : 1.3110
  Stricture           : 1.0000
Treino: 1097 | Validação: 235 | Teste: 236

DataLoaders prontos!
Batches treino: 35 | Validação: 8 | Teste: 8


In [17]:
import shutil

OUTPUT_DIR = "./data/dataset_processed"

for split, df_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    for _, row in df_split.iterrows():
        # Processa a imagem
        img = preprocess_image(row['image_path'], target_size=TARGET_SIZE, apply_norm=False)
        if img is None:
            continue

        # Cria pasta destino por classe
        dest_dir = os.path.join(OUTPUT_DIR, split, row['label'])
        os.makedirs(dest_dir, exist_ok=True)

        # Guarda
        filename = os.path.basename(row['image_path'])
        dest_path = os.path.join(dest_dir, filename)
        Image.fromarray(img.astype(np.uint8)).save(dest_path)

print("Dataset processado guardado em:", OUTPUT_DIR)

Dataset processado guardado em: ./data/dataset_processed
